In [0]:
class Pet:
    def __init__(self, name, species):
        self.name = name
        self.species = species

    def introduce(self):
        print(f'Hi, my name is {self.name} and I am a {self.species}')
              
dog = Pet('Buddy', 'Dog')
Cat = Pet('Garfield', 'Cat')

dog.introduce()
dog.color = 'Brown'
print(dog.color)

print(dog.name)
print(dog.species)
print(Cat.name)
print(Cat.species)

In [0]:
import pandas as pd
import os

class DataExtractor:
    def __init__(self, tables: dict):
        self.tables = tables

    def extract(self):
        data = {}
        for url, table_name in self.tables.items():
            if url.endswith('.csv'):
                data[table_name] = pd.read_csv(url, engine="python")
            elif url.endswith('.tsv'):
                data[table_name] = pd.read_csv(url, delimiter='\t', engine="python")
        return data
    
class DataTransformer:
    def __init__(self, columns_to_clean: dict):
        self.columns_to_clean = columns_to_clean

    def clean(self, data):
        for table_name, columns in self.columns_to_clean.items():
            for col in columns:
                if data[table_name][col].dtype == 'object':
                    data[table_name][col] = (
                        data[table_name][col]
                        .str.replace('$', '', regex=False)
                        .str.replace(',', '', regex=False)
                    )
                    data[table_name][col] = pd.to_numeric(data[table_name][col], errors='coerce')
        return data
    
class DataLoader:
    def __init__(self, destination_folder: str):
        self.destination_folder = destination_folder

    def load(self, data):
        os.makedirs(self.destination_folder, exist_ok=True)
        for table_name, df in data.items():
            path = os.path.join(self.destination_folder, f"{table_name}.csv")
            df.to_csv(path, index=False)

class ETLPipeline:
    def __init__(self, tables, columns_to_clean, folders):
        self.extractor = DataExtractor(tables)
        self.transformer = DataTransformer(columns_to_clean)
        self.raw_folder, self.clean_folder, self.transformed_folder = folders

    def run(self):
        print("Extracting data...")
        data = self.extractor.extract()

        print("Saving raw data...")
        DataLoader(self.raw_folder).load(data)

        print("Cleaning data...")
        cleaned = self.transformer.clean(data)
        DataLoader(self.clean_folder).load(cleaned)

        print("Transforming data...")
        # your transform_data() logic can go here
        transformed = cleaned  # placeholder
        DataLoader(self.transformed_folder).load(transformed)

        print("ETL complete!")

if __name__ == "__main__":
    TABLES = {
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/MovieLens_movies.csv": "movies_Id",
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/IMDb%20BoxOfficeMojo%20-%20Brands%20(US%20%26%20Canada).tsv": "brands_US_and_Canada",
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/IMDb%20BoxOfficeMojo%20-%20Brand_%20Marvel%20Comics.tsv": "brand_marvel_comics",
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/The%20Numbers%20-%20Domestic%20Box%20Office%20Daily%20-%20The%20Avengers.tsv": "Domestic_Box_Office_Daily_The_Avengers",
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/The%20Numbers%20-%20Domestic%20Box%20Office%20-%20Franchises.tsv": "Domestic_Box_Office_Franchises",
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/The%20Numbers%20-%20Domestic%20Box%20Office%20-%20Franchises%20-%20Marvel%20Cinematic%20Universe.tsv": "Domestic_Box_Office_Franchises_Marvel_Cinematic",
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/World%20Wide%20Box%20Office%20All%20Time%20Top%201000.tsv": "World_Wide_Box_Office_All_Time_Top_1000",
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/IMDb%20BoxOfficeMojo%20-%20Franchises%20(US%20%26%20Canada).tsv": "Franchises_us_and_Canada",
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/IMDb%20BoxOfficeMojo%20-%20Franchise_%20top20.tsv": "top_20_for_each_Franchise",
    "https://raw.githubusercontent.com/mansik95/IMDB-Analysis/master/Data/MovieLens_tags.csv": "tags"
    }

    COLUMNS_TO_CLEAN = {
    'Domestic_Box_Office_Franchises': ['Domestic_Box_Office', 'Infl_Adj_Dom_Box_Office', 'Worldwide_Box_Office'],
    'Domestic_Box_Office_Franchises_Marvel_Cinematic': ['Production_Budget', 'Opening_Weekend', 'Domestic_Box_Office', 'Worldwide_Box_Office'],
    'top_20_for_each_Franchise': ['Lifetime_Gross','Opening_Gross','Max_Theaters']
}

    FOLDERS = (
        "/Workspace/Users/nnoromnneoma96@gmail.com/Full-Stack-IMBD-Data-Analysis/Nneoma_Nnorom_Copy/src/raw",
        "/Workspace/Users/nnoromnneoma96@gmail.com/Full-Stack-IMBD-Data-Analysis/Nneoma_Nnorom_Copy/src/clean",
        "/Workspace/Users/nnoromnneoma96@gmail.com/Full-Stack-IMBD-Data-Analysis/Nneoma_Nnorom_Copy/src/transformed"
    )

    pipeline = ETLPipeline(TABLES, COLUMNS_TO_CLEAN, FOLDERS)
    pipeline.run()